In [ ]:

import getpass
from dotenv import load_dotenv
import pydantic
from crewai import LLM, Agent, Task, Crew
from com.example.agentic.agent.LLMManager import LLMManager


llm = LLMManager.create_llm('openai')
llm.call('Hello')

In [40]:
from crewai_tools import RagTool
from com.example.agentic.tools.config import _rag_tool_config
from crewai.rag.config.utils import set_rag_config, get_rag_client, clear_rag_config
from chromadb.utils.embedding_functions.openai_embedding_function import OpenAIEmbeddingFunction
from chromadb.config import Settings
from crewai.utilities.paths import db_storage_path

# ChromaDB (default)
from crewai.rag.chromadb.config import ChromaDBConfig

set_rag_config(ChromaDBConfig(
    batch_size = 100,
    embedding_function = OpenAIEmbeddingFunction(
        api_key=os.getenv("OPENAI_API_KEY"),
        model_name="nomic-embed-text:latest",
        api_base="http://localhost:11434/v1/",
        api_version="v1",
        api_type="ollama",
        api_key_env_var="OPENAI_API_KEY"
    )
))

chromadb_client = get_rag_client()

# Example operations (same API for any provider)
client = chromadb_client  # or chromadb_client



In [41]:
client.get_or_create_collection(collection_name="pdf-documents")

Collection(name=pdf-documents)

In [ ]:
import os 
from com.example.agentic.loader.LoadManager import LoadManager
# 
#client.create_collection(collection_name="pdf-documents")

work_dir = os.getenv("WORK_DIR")
_pdf_documents = [
    f"{work_dir}/docs/pdf/Advanced_Microservices.pdf",
    # f"{work_dir}/docs/pdf/Building Microservices - Designing Fine-Grained Systems.pdf",
    # f"{work_dir}/docs/pdf/Clean Architecture A Craftsman Guide to Software Structure and Design.pdf",
    # f"{work_dir}/docs/pdf/Data Science.pdf",
    # f"{work_dir}/docs/pdf/Designing_Data_Intensive_Applications.pdf",
    # f"{work_dir}/docs/pdf/Enterprise Integration Patterns - Designing, Building And Deploying Messaging.pdf",
    # f"{work_dir}/docs/pdf/Kafka_Streams_In_Action.pdf",
    # f"{work_dir}/docs/pdf/Microservices Best_Practices_for_Java.pdf",
    # f"{work_dir}/docs/pdf/Microservices_From_Day_one.pdf",
    # f"{work_dir}/docs/pdf/Microservices From Design to Deploy.pdf",
    # f"{work_dir}/docs/pdf/Microservices Patterns.pdf",
    # f"{work_dir}/docs/pdf/ML_book.pdf",
    # f"{work_dir}/docs/pdf/Software Architecture Patterns.pdf",
    # f"{work_dir}/docs/pdf/System Design-interview-an-insiders.pdf"
]

import uuid
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, length_function=len, separators=["\n\n", "\n", " ", ""])

for _pdf_doc in _pdf_documents :
    # Prepare data for ChromaDB
    _documents = LoadManager.from_pdfs([_pdf_doc])
    _documents_chunks = splitter.split_documents(_documents)
    if len(_documents_chunks):
        _docs_to_store = []
        for i,_d in enumerate(_documents_chunks) :
            _doc = dict()

            doc_id = f"pdf_{uuid.uuid4().hex[:8]}_{i}"
            _doc["id"] = doc_id
            _doc['content'] = _d.page_content

            # Prepare metadata
            _metadata = dict(_d.metadata)
            _metadata['doc_index'] = i
            _metadata['content_length'] = len(_d.page_content)
            #_metadata['content'] = _d.page_content
            _doc["metadata"] = _metadata
            
            # Document content
            _docs_to_store.append(_doc)
            
        # Add documents to vector store
        client.add_documents(
            collection_name="pdf-documents",
            documents=_docs_to_store
        )
    else :
        print("document chunks not provided.")

In [38]:
results = client.search(collection_name="pdf-documents", query="Advanced Microservices", limit=10)
for _result in results :
    print(_result)
    
#clear_rag_config()  # optional reset

AttributeError: 'dict' object has no attribute 'name'

In [ ]:
results = client.search(collection_name="pdf-documents", query="Kafaka Data Flow", limit=10)
for _result in results :
    print(_result)

#### RAG Tool Implementation

In [42]:
from crewai_tools import RagTool
from crewai_tools.tools.rag import RagToolConfig, VectorDbConfig, ProviderSpec
from crewai.rag.embeddings.providers.openai.types import OpenAIProviderSpec
# Create a RAG tool with custom configuration

vectordb: VectorDbConfig = {
    "provider": "chromadb",
    "config": {
        "collection_name": "pdf-documents",
        #"persist_directory":"/home/brijeshdhaker/IdeaProjects/crewai_design_document/storage", 
        "allow_reset": False, 
        "is_persistent": True
    }
}

embedding_model: OpenAIProviderSpec = {
    "provider": "openai",
    "config": {
        "model_name": "nomic-embed-text:latest",
        "organization_id":"sandbox",
        "api_base":"http://localhost:11434/v1",
        "api_version":"v1",
        "api_key":"ollama",
        "api_type":"ollama",
        "dimensions":768,
        "default_headers":{"X-Custom-Header": "ollama"}
    }
}

config: RagToolConfig = {
    "vectordb": vectordb,
    "embedding_model": embedding_model
}

rag_tool = RagTool(
    config=config, 
    collection_name="pdf-documents", 
    summarize=True,
    limit=1000
)


In [ ]:
from crewai_tools.rag.data_types import DataType

work_dir = os.getenv("WORK_DIR")
_pdf_documents = [
    f"{work_dir}/docs/pdf/Advanced_Microservices.pdf",
    # f"{work_dir}/docs/pdf/Building Microservices - Designing Fine-Grained Systems.pdf",
    # f"{work_dir}/docs/pdf/Clean Architecture A Craftsman Guide to Software Structure and Design.pdf",
    # f"{work_dir}/docs/pdf/Data Science.pdf",
    # f"{work_dir}/docs/pdf/Designing_Data_Intensive_Applications.pdf",
    # f"{work_dir}/docs/pdf/Enterprise Integration Patterns - Designing, Building And Deploying Messaging.pdf",
    # f"{work_dir}/docs/pdf/Kafka_Streams_In_Action.pdf",
    # f"{work_dir}/docs/pdf/Microservices Best_Practices_for_Java.pdf",
    # f"{work_dir}/docs/pdf/Microservices_From_Day_one.pdf",
    # f"{work_dir}/docs/pdf/Microservices From Design to Deploy.pdf",
    # f"{work_dir}/docs/pdf/Microservices Patterns.pdf",
    # f"{work_dir}/docs/pdf/ML_book.pdf",
    #f"{work_dir}/docs/pdf/Software Architecture Patterns.pdf",
    #f"{work_dir}/docs/pdf/System Design-interview-an-insiders.pdf"
]
for pdf_file in _pdf_documents :
    # Add content from a file
    rag_tool.add(data_type=DataType.PDF_FILE, path=pdf_file)

# Add a txt file
rag_tool.add(data_type=DataType.MDX, path=f"{work_dir}/docs/text/Brijesh_K_Dhaker.md")

# Add a web page
rag_tool.add(data_type=DataType.WEBSITE, url="https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/rag/rag-solution-design-and-evaluation-guide")

# Add a YouTube video
#rag_tool.add(data_type=DataType.YOUTUBE_VIDEO, url="https://www.youtube.com/watch?v=VIDEO_ID")

# Add a directory of files
#rag_tool.add(data_type=DataType.DIRECTORY, path=f"{work_dir}/docs")

In [43]:
_rag_results = rag_tool.run(query="Kafaka Data Flow")
print(_rag_results)    

Relevant Content:
Patches Welcome ��������������������������������������������������������������������������������169
References �������������������������������������������������������������������������������������171
Index ����������������������������������������������������������������������������������������������175

Error Reporting �������������������������������������������������������������������������������������������������������40
Responses Should Mimic Requests ������������������������������������������������������������������������43
HTTP Headers ���������������������������������������������������������������������������������������������������������44
API Standards ����������������������������������������������������������������������������������������44
Simple Response Envelope ������������������������������������������������������������������������������������45
JSON Schema ��������������������������������������������������������������������������������������������������

In [45]:
from crewai.knowledge.storage.knowledge_storage import KnowledgeStorage
from crewai.knowledge.source.base_knowledge_source import BaseKnowledgeSource
from com.example.agentic.tools.config import _rag_tool_config
import requests
from datetime import datetime
from typing import Dict, Any
from pydantic import BaseModel, Field


class DesignKnowledgeSource(BaseKnowledgeSource):
    """Knowledge source that fetches data from Vectorstore."""
    limit: int = Field(default=10, description="Number of articles to fetch")
    
    def load_content(self) -> Dict[Any, str]:
        """Fetch and format space news articles."""
        try:
            self._chunk_text("Hello")
            self._save_documents()
            
            self.storage.search()
            response = requests.get(
                f"{self.api_endpoint}?limit={self.limit}"
            )
            response.raise_for_status()

            data = response.json()
            articles = data.get('results', [])

            formatted_data = self.validate_content(articles)
            return {self.api_endpoint: formatted_data}
        except Exception as e:
            raise ValueError(f"Failed to fetch space news: {str(e)}")

    def validate_content(self, articles: list) -> str:
        """Format articles into readable text."""
        formatted = "Space News Articles:\n\n"
        for article in articles:
            formatted += f"""
                Title: {article['title']}
                Published: {article['published_at']}
                Summary: {article['summary']}
                News Site: {article['news_site']}
                URL: {article['url']}
                -------------------"""
        return formatted

    def add(self) -> None:
        """Process and store the articles."""
        content = self.load_content()
        for _, text in content.items():
            chunks = self._chunk_text(text)
            self.chunks.extend(chunks)

        self._save_documents()

    def aadd(self) -> None:
        return super().aadd()        

In [46]:
from crewai.knowledge.storage.knowledge_storage import KnowledgeStorage
from com.example.agentic.tools.config import _rag_tool_config

# Create custom storage with specific embedder
design_knowledge_storage = KnowledgeStorage(
    embedder=_rag_tool_config['embedder'],
    collection_name="pdf-documents"
)

# Create knowledge source
design_knowledge_source = DesignKnowledgeSource(
    collection_name="pdf-documents",
    storage=design_knowledge_storage,
    limit=10,
)

In [ ]:
from crewai.tools import tool
from crewai_tools import PDFSearchTool
from com.example.agentic.tools.config import _tool_config, _rag_tool_config
#pydantic.SkipValidation

# 1. Initialize the tool
# - embedding_model (required): choose provider + provider-specific config
# - vectordb (required): choose vector DB and pass its config
#@tool

from crewai_tools import RagTool
from crewai_tools.tools.rag import RagToolConfig, VectorDbConfig, ProviderSpec
from crewai.rag.embeddings.providers.openai.types import OpenAIProviderSpec
# Create a RAG tool with custom configuration

vectordb: VectorDbConfig = {
    "provider": "chromadb",
    "config": {
        "collection_name": "pdf-documents",
        "persist_directory":"/home/brijeshdhaker/IdeaProjects/crewai_design_document/storage", 
        "allow_reset": False, 
        "is_persistent": True
    }
}

embedding_model: OpenAIProviderSpec = {
    "provider": "openai",
    "config": {
        "model_name": "nomic-embed-text:latest",
        "organization_id":"sandbox",
        "api_base":"http://localhost:11434/v1",
        "api_version":"v1",
        "api_key":"ollama",
        "api_type":"ollama",
        "dimensions":768,
        "default_headers":{"X-Custom-Header": "ollama"}
    }
}

config: RagToolConfig = {
    "vectordb": vectordb,
    "embedding_model": embedding_model
}

_pdf_search_tool = PDFSearchTool(
    config=config,
    collection_name="pdf-documents", 
    summarize=True,
    limit=1000
)


In [ ]:
work_dir = os.getenv("WORK_DIR")
_pdf_documents = [
    f"{work_dir}/docs/pdf/Advanced_Microservices.pdf",
    # f"{work_dir}/docs/pdf/Building Microservices - Designing Fine-Grained Systems.pdf",
    # f"{work_dir}/docs/pdf/Clean Architecture A Craftsman Guide to Software Structure and Design.pdf",
    # f"{work_dir}/docs/pdf/Data Science.pdf",
    # f"{work_dir}/docs/pdf/Designing_Data_Intensive_Applications.pdf",
    # f"{work_dir}/docs/pdf/Enterprise Integration Patterns - Designing, Building And Deploying Messaging.pdf",
    # f"{work_dir}/docs/pdf/Kafka_Streams_In_Action.pdf",
    # f"{work_dir}/docs/pdf/Microservices Best_Practices_for_Java.pdf",
    # f"{work_dir}/docs/pdf/Microservices_From_Day_one.pdf",
    # f"{work_dir}/docs/pdf/Microservices From Design to Deploy.pdf",
    # f"{work_dir}/docs/pdf/Microservices Patterns.pdf",
    # f"{work_dir}/docs/pdf/ML_book.pdf",
    #f"{work_dir}/docs/pdf/Software Architecture Patterns.pdf",
    #f"{work_dir}/docs/pdf/System Design-interview-an-insiders.pdf"
]

for pdf_file in _pdf_documents :
    _pdf_search_tool.add(pdf=pdf_file)


In [ ]:
results = _pdf_search_tool.run(query="Spark & Kafka")
results

In [ ]:
from datetime import datetime
from crewai import Crew, Agent, Task, Process
from crewai import context
from emoji import config
from com.example.agentic.tools.config import _tool_config

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
print(run_id)

from crewai_tools import FileReadTool

# Initialize tool
file_read_tool = FileReadTool(file_path='/home/brijeshdhaker/IdeaProjects/crewai_design_document/docs/text/Brijesh_Resume.txt')
#file_read_tool.run()

# 3. Agent and Task Definition
text_specialist = Agent(
    role='Text File Reader',
    goal='Extract candidate skills and identifying experience, company and work location',
    backstory='You as expert resume analyst and pull candidate skills and identifying experience, company and work location.',
    tools=[file_read_tool],
    #llm=llm
)

# 2. Define the Agent
resume_analyst = Agent(
    role='As a resume analyst',
    goal='Extract candidate skills and identifying experience, company and work location',
    backstory='You as expert resume analyst and pull candidate skills and identifying experience, company and work location.',
    tools=[],
    allow_delegation=False,
    verbose=True,
    #llm=llm
)

# 3. Define the Task
read_task = Task(
    description='Read text content',
    expected_output='text contain in the file.',
    agent=text_specialist,
    output_file=f'outputs/L004/read_task_{run_id}.txt'
)

extract_task = Task(
    description='review the content that you receive and make sure you extract skills, experience and work location',
    expected_output='A json string contain skills, experience and work location and list of domain knowledge',
    agent=text_specialist,
    context=[read_task],
    output_file=f'outputs/L004/extract_task_{run_id}.json'
)

# 4. Run the Crew
crew = Crew(
    agents=[text_specialist], 
    tasks=[read_task,extract_task],
    process=Process.sequential,
    verbose=True,
    #planning=True,
    #planning_llm="openai/llama3.2:3b-instruct-q8_0"
)
result = crew.kickoff()
print(result)

In [ ]:
def generate_paper_summary(text):

    try:

        llm = OpenAI(temperature=0.3, max_tokens=1000)  

        summary_prompt = (
            "You are an expert research analyst. Please provide a comprehensive summary of this research paper. "
            "Include the following sections:\n\n"
            "1. **Title and Authors** (if available)\n"
            "2. **Abstract/Summary** - Main research question and objectives\n"
            "3. **Methodology** - How the research was conducted\n"
            "4. **Key Findings** - Main results and discoveries\n"
            "5. **Contributions** - What new knowledge or insights this paper provides\n"
            "6. **Limitations** - Any limitations mentioned by the authors\n"
            "7. **Future Work** - Suggested future research directions\n\n"
            "Please be thorough but concise. Use clear headings and bullet points where appropriate.\n\n"
            f"Research Paper Text:\n{text[:8000]}\n\n"  # Limit text to avoid token limits
            "Summary:"
        )
        summary = llm.invoke(summary_prompt)
        return summary
    except Exception as e:
        st.error(f"Error generating summary: {str(e)}")
        return None

In [ ]:
def cleanup_chroma_db():
    try:
        if os.path.exists("./chroma_db"):
            shutil.rmtree("./chroma_db")
    except Exception as e:
        st.warning(f"Could not clean up ChromaDB directory: {str(e)}")

def process_document(text):
    try:
        cleanup_chroma_db()
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=100,
            length_function=len,
        )

        docs = splitter.create_documents([text])
        try:
            vectorstore = Chroma.from_documents(
                documents=docs,
                embedding=OpenAIEmbeddings(),
                collection_name="research_papers"
            )
            return vectorstore

        except Exception as chroma_error:
            st.warning(f"ChromaDB failed, trying FAISS: {str(chroma_error)}")
            vectorstore = FAISS.from_documents(
                documents=docs,
                embedding=OpenAIEmbeddings()
            )
            return vectorstore            

    except Exception as e:
        st.error(f"Error processing document: {str(e)}")
        cleanup_chroma_db()
        return None

In [ ]:
def retrieval_action(question, vectorstore):
    results = vectorstore.similarity_search(question, k=5)
    return "\n\n".join([f"Passage {i+1}: {doc.page_content}" for i, doc in enumerate(results)])

def generation_action(inputs):
    if isinstance(inputs, dict):
        question = inputs.get("user", "")
        context = inputs.get("Retriever", "")
    else:
        question = "User question not found in inputs"
        context = str(inputs)
    prompt = (
        "You are an expert research analyst. You have been given a specific research paper and asked a question about it. "
        "Based on the retrieved passages from the research paper, provide a detailed answer with numbered citations. "
        "Use only the information from the provided passages to answer the question.\n\n"
        f"USER QUESTION: {question}\n\n"
        f"RELEVANT PASSAGES FROM THE RESEARCH PAPER:\n{context}\n\n"
        "INSTRUCTIONS:\n"
        "1. Answer the question based ONLY on the provided passages from the research paper\n"
        "2. Use numbered citations [1], [2], etc. for each key point you reference\n"
        "3. If the passages don't contain enough information to answer, say so clearly\n"
        "4. Be specific and reference the actual content from the paper\n"
        "5. Focus on the key findings, results, and conclusions mentioned in the passages\n"
        "6. If asked about key findings, look for results, outcomes, discoveries, or conclusions\n\n"
        "ANSWER:"
    )    
    
    llm = OpenAI(model = 'gpt-3.5-turbo-instruct' , temperature=0.2, max_tokens=1500)
    return llm.invoke(prompt)

def create_crew(vectorstore, status_callback=None):

    def retrieval_wrapper(question):
        if status_callback:
            status_callback("Searching for relevant passages in the research paper...")

        result = retrieval_action(question, vectorstore)
        if status_callback:
            status_callback("Found relevant passages for analysis")

        return result

    def generation_wrapper(inputs):
        if status_callback:
            status_callback("Analyzing retrieved passages and generating comprehensive answer...")

        result = generation_action(inputs)
        if status_callback:
            status_callback("Generated detailed answer with citations")

        return result

    retriever = Agent(
        role="Retriever",
        goal="Retrieve relevant passages from the research paper",
        backstory="You are an expert at finding relevant information in research papers",
        action=retrieval_wrapper,
        verbose=True
    )

    generator = Agent(
        role="Generator", 
        goal="Generate comprehensive answers based on retrieved passages",
        backstory="You are an expert research analyst who provides detailed, citation-based answers",
        action=generation_wrapper,
        verbose=True
    )

    retrieval_task = Task(
        description="Retrieve relevant passages from the research paper for the user's question",
        agent=retriever,
        expected_output="Relevant passages from the research paper that can answer the user's question"
    )

    generation_task = Task(
        description="Generate a comprehensive answer with citations based on the retrieved passages from the research paper",
        agent=generator,
        expected_output="A detailed answer with numbered citations based on the research paper content",
        context=[retrieval_task]
    )    

    crew = Crew(
        agents=[retriever, generator],
        tasks=[retrieval_task, generation_task],
        verbose=True
    )

    return crew

In [ ]:
def main():

    st.title("Research Paper Analyst")
    st.markdown("Upload a research paper and ask questions about it using AI-powered analysis.")    
    with st.sidebar:
        st.header("Document Upload")
        uploaded_file = st.file_uploader(
            "Choose a PDF file",
            type="pdf",
            help="Upload a research paper in PDF format"
        )        

        if uploaded_file is not None:
            st.info(f"File uploaded: {uploaded_file.name}")        
            if st.button("Process Document", type="primary"):
                with st.spinner("Processing document and generating summary..."):
                    text = extract_text_from_pdf(uploaded_file)    
                    if text:
                        summary = generate_paper_summary(text)        
                        if summary:
                            st.session_state.paper_summary = summary
                            vectorstore = process_document(text)        
                        if vectorstore:
                            st.session_state.vectorstore = vectorstore
                            st.session_state.document_processed = True
                            st.info("Document processed successfully! Summary generated and ready for questions.")
                        else:
                            st.error("Failed to process document.")
                    else:
                        st.error("Failed to extract text from PDF.")

    if not st.session_state.document_processed:
        st.info("Please upload and process a PDF document using the sidebar to get started.")   

    else:
        with st.expander("Paper Summary", expanded=False):
            if st.session_state.paper_summary:
                st.markdown(st.session_state.paper_summary)
            else:
                st.warning("Summary not available.")

        st.subheader("Ask Questions About Your Research Paper")        

        for message in st.session_state.chat_history:
            with st.chat_message(message["role"]):
                st.write(message["content"]) 

        if prompt := st.chat_input("Ask a question about the research paper..."):
            st.session_state.chat_history.append({"role": "user", "content": prompt})  

            with st.chat_message("user"):
                st.write(prompt)        

            with st.chat_message("assistant"):
                progress_container = st.container()
                execution_trace_container = st.expander("Execution Details", expanded=False)     

                with progress_container:
                    st.info("Initializing AI agents...")

                try:
                    status_placeholder = st.empty()
                    trace_placeholder = execution_trace_container.empty()
                    execution_steps = []

                    def log_step(step):
                        execution_steps.append(step)
                        with trace_placeholder:
                            for i, s in enumerate(execution_steps, 1):
                                st.write(f"{i}. {s}")

                    def status_callback(message):
                        status_placeholder.info(message)
                        log_step(message)                    

                    status_placeholder.info("Initializing AI agents...")
                    log_step("CrewAI agents initialized")
                    crew = create_crew(st.session_state.vectorstore, status_callback)
                    status_placeholder.info("Starting AI analysis...")
                    log_step("CrewAI execution started")
                    status_placeholder.info("Retrieving relevant passages...")
                    retrieved_passages = retrieval_action(prompt, st.session_state.vectorstore)
                    log_step("Retrieved relevant passages from the research paper")
                    status_placeholder.info("Generating comprehensive answer...")

                    inputs = {
                        "user": prompt,
                        "Retriever": retrieved_passages
                    }

                    response = generation_action(inputs)
                    log_step("Generated detailed answer with citations")
                    status_placeholder.info("Analysis complete!")
                    st.session_state.chat_history.append({"role": "assistant", "content": response})
                    st.markdown("### AI Analysis Result:")
                    st.write(response)
                    with execution_trace_container:
                        st.markdown("### Execution Summary:")
                        st.info("**Retriever Agent**: Found relevant passages from the research paper")
                        st.info("**Generator Agent**: Created comprehensive answer with citations")
                        st.info(f"**Total Steps**: {len(execution_steps)}")
                        st.markdown("### Detailed Execution Trace:")

                        for i, step in enumerate(execution_steps, 1):
                            st.write(f"{i}. {step}")            

                except Exception as e:
                    error_msg = f"Error generating response: {str(e)}"
                    st.error(error_msg)
                    st.session_state.chat_history.append({"role": "assistant", "content": error_msg})

        

        if st.button("Clear Chat History"):
            st.session_state.chat_history = []
            st.rerun()


‍if __name__ == "__main__":
    main()

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.knowledge.storage.knowledge_storage import KnowledgeStorage
from crewai.knowledge.source.base_knowledge_source import BaseKnowledgeSource
from com.example.agentic.tools.config import _rag_tool_config
import requests
from datetime import datetime
from typing import Dict, Any
from pydantic import BaseModel, Field

# Create custom storage with specific embedder
space_knowledge_storage = KnowledgeStorage(
    #embedder=_rag_tool_config['embedder'],
    collection_name="space_knowledge_storage"
)

class SpaceNewsKnowledgeSource(BaseKnowledgeSource):
    """Knowledge source that fetches data from Space News API."""

    api_endpoint: str = Field(description="API endpoint URL")
    limit: int = Field(default=10, description="Number of articles to fetch")

    def load_content(self) -> Dict[Any, str]:
        """Fetch and format space news articles."""
        try:
            response = requests.get(
                f"{self.api_endpoint}?limit={self.limit}"
            )
            response.raise_for_status()

            data = response.json()
            articles = data.get('results', [])

            formatted_data = self.validate_content(articles)
            return {self.api_endpoint: formatted_data}
        except Exception as e:
            raise ValueError(f"Failed to fetch space news: {str(e)}")

    def validate_content(self, articles: list) -> str:
        """Format articles into readable text."""
        formatted = "Space News Articles:\n\n"
        for article in articles:
            formatted += f"""
                Title: {article['title']}
                Published: {article['published_at']}
                Summary: {article['summary']}
                News Site: {article['news_site']}
                URL: {article['url']}
                -------------------"""
        return formatted

    def add(self) -> None:
        """Process and store the articles."""
        content = self.load_content()
        for _, text in content.items():
            chunks = self._chunk_text(text)
            self.chunks.extend(chunks)

        self._save_documents()

    def aadd(self) -> None:
        return super().aadd()    
    

# Create knowledge source
recent_news = SpaceNewsKnowledgeSource(
    api_endpoint="https://api.spaceflightnewsapi.net/v4/articles",
    limit=10,
    storage=space_knowledge_storage
)
#recent_news.storage = space_knowledge_storage


# Create specialized agent
space_analyst = Agent(
    role="Space News Analyst",
    goal="Answer questions about space news accurately and comprehensively",
    backstory="""You are a space industry analyst with expertise in space exploration,
    satellite technology, and space industry trends. You excel at answering questions
    about space news and providing detailed, accurate information.""",
    knowledge_sources=[recent_news],
    #llm=LLM(model="ollama/llama3.2:3b-instruct-q8_0", base_url="http://localhost:11434", temperature=0.0)
)

# Create task that handles user questions
analysis_task = Task(
    description="Answer this question about space news: {user_question}",
    expected_output="A detailed answer based on the recent space news articles",
    agent=space_analyst
)

# Create and run the crew
crew = Crew(
    agents=[space_analyst],
    tasks=[analysis_task],
    verbose=True,
    process=Process.sequential,
    embedder=_rag_tool_config['embedder']
)

# Example usage
result = crew.kickoff(
    inputs={"user_question": "What are the latest developments in space exploration?"}
)